# # pydbsp tutorial

This is a condensed walkthrough of the core API: `ZSet`, streams,
incremental joins (nested-loop and sort-merge), and a small
batch-vs-incremental benchmark.

## 1. Z-sets and joins

A Z-set is a weighted multiset — `Dict[value, int]`. Joins are
pointwise predicates with a projection.

In [1]:
from pydbsp.zset import ZSet
from pydbsp.zset.functions.bilinear import join

employees = ZSet({(0, "kristjan"): 1, (1, "mark"): 1, (2, "mike"): 1})
salaries = ZSet({(0, "38750"): 1, (1, "50000"): 1, (2, "40000"): 1})

result = join(
    employees,
    salaries,
    lambda l, r: l[0] == r[0],
    lambda l, r: (l[0], l[1], r[1]),
)
print(result.inner)

{(0, 'kristjan', '38750'): 1, (1, 'mark', '50000'): 1, (2, 'mike', '40000'): 1}


## 2. Streams and pull-based observation

Every stream is a function `Time → Value` over some lattice. You
drive external inputs via `Input.push(t, value)` and read results
through a `History` cursor.

In [2]:
from pydbsp.stream import Input
from pydbsp.core import Time1D
from pydbsp.zset import ZSetAddition
from pydbsp.history import History

group = ZSetAddition[tuple[int, str]]()
emp_in = Input(group, Time1D)
emp_in.push((0,), employees)

h = History(emp_in)
h.try_step()
print("emp_in at tick 0:", h.at((0,)).inner)

emp_in at tick 0: {(0, 'kristjan'): 1, (1, 'mark'): 1, (2, 'mike'): 1}


## 3. Incremental join — 4-term DLD

`DLDJoin` is the doubly-incremental nested-loop join on the flat
2-D product lattice (axis 0 = outer, axis 1 = inner). Inputs are
1-D streams lifted to 2-D via `TimeAxisIntroduction`; outputs are
read by `Evaluator` at a concrete `(outer, inner)` cell.

In [ ]:
from pydbsp.core import Time2D
from pydbsp.evaluator import Evaluator
from pydbsp.stream.functions.linear import TimeAxisIntroduction
from pydbsp.stream.zset.operators.bilinear import DLDJoin

lattice = Time2D
emp_group = ZSetAddition[tuple[int, str]]()
sal_group = ZSetAddition[tuple[int, str]]()
out_group = ZSetAddition[tuple[int, str, str]]()

emp_input = Input(emp_group, Time1D)
sal_input = Input(sal_group, Time1D)
emp_2d = TimeAxisIntroduction(emp_input, emp_group, lattice, axis=1)
sal_2d = TimeAxisIntroduction(sal_input, sal_group, lattice, axis=1)

incr_join = DLDJoin(
    emp_2d,
    sal_2d,
    lambda l, r: l[0] == r[0],
    lambda l, r: (l[0], l[1], r[1]),
    out_group,
    lattice,
)

emp_input.push((0,), employees)
sal_input.push((0,), salaries)

ev = Evaluator(incr_join)
print("incremental join at (0, 0):", ev.at((0, 0)).inner)

## 4. Sort-merge indexed join

When both sides share a join key, sort-merge is asymptotically faster
than nested-loop for dense outputs. Wrap each lifted input in
`Index` to build an indexed Z-set, then feed to `DLDSortMergeJoin`.

In [ ]:
from pydbsp.indexed_zset.operators.bilinear import DLDSortMergeJoin
from pydbsp.indexed_zset.operators.linear import Index

emp_idx = Index(emp_2d, lambda e: e[0], emp_group)
sal_idx = Index(sal_2d, lambda s: s[0], sal_group)

smj = DLDSortMergeJoin(
    emp_idx,
    sal_idx,
    lambda key, l, r: (key, l[1], r[1]),
    out_group,
    lattice,
)

ev_smj = Evaluator(smj)
print("sort-merge join at (0, 0):", ev_smj.at((0, 0)).inner)

## 5. Reachability end-to-end

`IncrementalReachability` computes transitive closure over an edge
`ZSet` via feedback. Push edges, `saturate_reach`, read observable.

In [5]:
from pydbsp.algorithms import IncrementalReachability, saturate_reach

edges = ZSet({(0, 1): 1, (1, 2): 1, (2, 3): 1})
reach = IncrementalReachability()
reach.edges.push((0,), edges)
saturate_reach(reach)
h = History(reach.observable)
h.try_step()
print("TC pairs:", sorted(h.at((0,)).inner))

TC pairs: [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]


## 6. Datalog end-to-end

Same program as `rdfs.ipynb`'s Datalog comparison, but minimal: a
single TC rule over a tiny EDB.

In [6]:
from pydbsp.algorithms import IncrementalDatalog, saturate, Variable

program = ZSet(
    {
        (
            ("T", (Variable("X"), Variable("Y"))),
            ("E", (Variable("X"), Variable("Y"))),
        ): 1,
        (
            ("T", (Variable("X"), Variable("Z"))),
            ("E", (Variable("X"), Variable("Y"))),
            ("T", (Variable("Y"), Variable("Z"))),
        ): 1,
    }
)
edb = ZSet({("E", (0, 1)): 1, ("E", (1, 2)): 1, ("E", (2, 3)): 1})

dl = IncrementalDatalog()
dl.edb.push((0,), edb)
dl.program.push((0,), program)
saturate(dl)
h = History(dl.observable)
h.try_step()
print("Datalog derived T pairs:", sorted(k[1] for k in h.at((0,)).inner if k[0] == "T"))

Datalog derived T pairs: [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
